In [ ]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.neighbors import LocalOutlierFactor
# from sklearn.preprocessing import StandardScaler
# from datetime import timedelta

# # --- Configuration (بدون تغییر) ---
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_clustering\clustering\dsas_g11_lubrication\clustering_g11_lubrication_output3.xlsx'
# target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# def run_smart_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ File not found: {file_path}")
#         return

#     # 1. Data Loading & Preprocessing
#     df = pd.read_excel(file_path)
#     if 'date' in df.columns:
#         df['date'] = pd.to_datetime(df['date'])

#     df_model = df.dropna(subset=target_sensors).copy()

#     # 2. Denoising
#     for sensor in target_sensors:
#         df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()

#     smooth_cols = [f'{s}_smooth' for s in target_sensors]
#     df_model = df_model.dropna(subset=smooth_cols).copy()
#     df_model = df_model.reset_index(drop=True)

#     # 3. Feature Scaling
#     scaler = StandardScaler()
#     scaled_data = scaler.fit_transform(df_model[smooth_cols])

#     # 4. Applying Local Outlier Factor (LOF)
#     # n_neighbors: تعداد همسایگان برای مقایسه چگالی محلی
#     # contamination: نسبت تخمینی داده‌های ناهنجار (در اینجا 0.05 یا 5 درصد فرض شده)
#     lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

#     # تشخیص وضعیت (1 برای نرمال، -1 برای ناهنجار)
#     df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)

#     # استخراج نمرات منفی فاکتور خارج‌افتادگی (هرچه کمتر و منفی‌تر باشد، داده ناهنجارتر است)
#     # برای هماهنگی با منطق کد قبلی، آن را مثبت و معکوس می‌کنیم تا به عنوان Degradation_Index استفاده شود
#     lof_scores = -lof.negative_outlier_factor_
#     df_model['Degradation_Index'] = lof_scores

#     # 5. Standardized Health Labeling
#     # در LOF نمرات نزدیک به 1 نرمال هستند و نمرات بزرگتر (مثلاً بالای 1.5) نشان دهنده انحراف هستند
#     def get_health_status(row):
#         if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
#             return "Investigation Needed (Operational Drift)"
#         elif row['Degradation_Index'] > 1.3:
#             return "Observation Required (Pattern Change)"
#         else:
#             return "Healthy (Optimal Performance)"

#     df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)

#     # 6. Filtering for the Last 30 Days
#     if 'date' in df_model.columns:
#         last_date = df_model['date'].max()
#         final_output = df_model[df_model['date'] >= (last_date - timedelta(days=30))].copy()
#     else:
#         final_output = df_model

#     # 7. Exporting to Excel
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)
#         final_output.to_excel(output_filename, index=False)
#         print("🚀 LOF Analysis completed successfully.")
#         print(f"📊 Outlier Factors calculated based on local density deviations.")

#         print(f"📁 Output saved: {output_filename}")
#     except Exception as e:
#         print(f"❌ Error saving file: {e}")

# run_smart_analysis()

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

def run_smart_analysis():
    """اجرای تحلیل با Local Outlier Factor (LOF) برای سیستم روغن‌کاری و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # --- Configuration ---
    file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_lubrication_system_clustering\clustering\dsas_g11_lubrication\clustering_g11_lubrication_output3.xlsx'
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return None

    # 1. Data Loading & Preprocessing
    print("🔄 مرحله 1: بارگذاری و پیش‌پردازش داده...")
    df = pd.read_excel(file_path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

    df_model = df.dropna(subset=target_sensors).copy()
    print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")

    # 2. Denoising
    print("🔄 مرحله 2: حذف نویز با میانگین متحرک...")
    for sensor in target_sensors:
        df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()

    smooth_cols = [f'{s}_smooth' for s in target_sensors]
    df_model = df_model.dropna(subset=smooth_cols).copy()
    df_model = df_model.reset_index(drop=True)
    print(f"✅ پس از حذف نویز: {len(df_model):,} رکورد")

    # 3. Feature Scaling
    print("🔄 مرحله 3: استانداردسازی داده‌ها...")
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[smooth_cols])
    print(f"✅ استانداردسازی با {len(smooth_cols)} سنسور انجام شد")

    # 4. Applying Local Outlier Factor (LOF)
    print("🔄 مرحله 4: اعمال الگوریتم Local Outlier Factor (LOF)...")
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

    # تشخیص وضعیت (1 برای نرمال، -1 برای ناهنجار)
    df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)

    # استخراج نمرات منفی فاکتور خارج‌افتادگی
    lof_scores = -lof.negative_outlier_factor_
    df_model['Degradation_Index'] = lof_scores
    
    anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
    print(f"   تعداد ناهنجاری‌های شناسایی شده: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
    print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")

    # 5. Standardized Health Labeling
    print("🔄 مرحله 5: لیبل‌گذاری وضعیت سلامت...")
    
    def get_health_status(row):
        if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 1.3:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

    df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
    
    # نمایش توزیع وضعیت‌ها
    status_counts = df_model['Health_Status'].value_counts()
    print(f"\n📊 توزیع وضعیت‌ها:")
    for status, count in status_counts.items():
        print(f"   {status}: {count:,} ({count/len(df_model)*100:.2f}%)")

    # 6. Filtering for the Last 30 Days
    print("🔄 مرحله 6: فیلتر کردن داده‌های ۳۰ روز آخر...")
    if 'date' in df_model.columns:
        last_date = df_model['date'].max()
        start_date = last_date - timedelta(days=30)
        final_output = df_model[df_model['date'] >= start_date].copy()
        print(f"   بازه خروجی: {start_date} تا {last_date}")
        print(f"   تعداد رکوردهای ۳۰ روز آخر: {len(final_output):,}")
    else:
        final_output = df_model
        print("   ⚠️ ستون 'date' وجود ندارد، تمام داده‌ها ذخیره می‌شوند.")

    # نمایش آمار نهایی
    print(f"\n📊 آمار نهایی:")
    print(f"   کل رکوردها: {len(df_model):,}")
    print(f"   رکوردهای ۳۰ روز آخر: {len(final_output):,}")
    
    # نمایش نمونه‌هایی که نیاز به بررسی دارند
    investigation_needed = final_output[final_output['Health_Status'].str.contains('Investigation Needed')]
    if len(investigation_needed) > 0:
        print(f"\n⚠️ تعداد رکوردهای نیازمند بررسی: {len(investigation_needed):,}")
        print(f"   درصد نیازمند بررسی: {len(investigation_needed)/len(final_output)*100:.2f}%")

    # 7. Exporting to Excel
    print("💾 مرحله 7: ذخیره خروجی...")
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ LOF Analysis completed successfully.")
        print(f"📊 Outlier Factors calculated based on local density deviations.")
        print(f"📁 Output saved: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(final_output):,}")
        print(f"📋 تعداد ستون‌ها: {len(final_output.columns)}")
    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return final_output

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی سیستم روغن‌کاری با LOF")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:14", "22:15", "22:22"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_smart_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل خوشه‌بندی سیستم روغن‌کاری با LOF")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه تحلیل خوشه‌بندی سیستم روغن‌کاری با LOF
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی سیستم روغن‌کاری با LOF
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-02 22:22:08
🔄 شروع تحلیل در 2026-07-02 22:22:08
🔄 مرحله 1: بارگذاری و پیش‌پردازش داده...
✅ داده بارگذاری شد. تعداد رکوردها: 7,117
🔄 مرحله 2: حذف نویز با میانگین متحرک...
✅ پس از حذف نویز: 7,113 رکورد
🔄 مرحله 3: استانداردسازی داده‌ها...
✅ استانداردسازی با 7 سنسور انجام شد
🔄 مرحله 4: اعمال الگوریتم Local Outlier Factor (LOF)...
   تعداد ناهنجاری‌های شناسایی شده: 356 (5.00%)
   محدوده شاخص تخریب: 0.9298 تا 8.9552
🔄 مرحله 5: لیبل‌گذاری وضعیت سلامت...

📊 توزیع وضعیت‌ها:
   Healthy (Optimal Performance): 6,498 (91.35%)
   Investigation Needed (Operational Drift): 356 (5.00%)
   Observation Required (Pattern Change): 259 (3.64%)
🔄 مرحله 6: فیلتر کردن داده‌های ۳۰ روز آخر...
   بازه خروجی: 2026-04-08 20:38:09 تا 2026-05-08